# Community Detection và EDA

Notebook này chạy hoặc tái sử dụng Leiden trên toàn graph, phân tích cấu trúc community và mức độ tập trung fraud, sau đó tạo community features cho thí nghiệm với TGAT.


In [1]:
from __future__ import annotations

from contextlib import AbstractContextManager
from datetime import datetime
from pathlib import Path
import gc
import hashlib
import json
import platform
import random
import threading
import time

import igraph as ig
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "dgraphfin.npz"
RESULT_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_results.json"
SCHEMA_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_results.schema.json"
ASSIGNMENT_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_assignments.npz"
ASSIGNMENT_MANIFEST_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_assignments.manifest.json"
COMMUNITY_TABLE_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_table.csv.gz"
RISK_SELECTION_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_risky_community_selection.json"
RISKY_TABLE_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_risky_communities.csv"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures" / "sprint4"
COMMUNITY_FEATURE_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_features.npz"
COMMUNITY_FEATURE_MANIFEST_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "sprint4_community_features.manifest.json"

EXPECTED_DATA_SHA256 = "95470dab2c48523f7118a92204c090de37a957bb053bd5841c7bdba09558ba85"
EXPECTED_NODE_COUNT = 3_700_550
EXPECTED_DIRECTED_EDGE_COUNT = 4_300_999
EXPECTED_STRUCTURAL_ARC_COUNT = 7_994_520
COMMUNITY_SEED = 42
LEIDEN_RESOLUTION = 1.0
LEIDEN_ITERATIONS = 2
WRITE_ARTIFACTS = True
REUSE_LOCKED_ASSIGNMENT = True
MIN_COMMUNITY_SIZE = 20
MIN_LABELED_COMMUNITY = 20
MIN_TRAIN_FRAUD_COUNT = 2
MIN_TRAIN_FRAUD_LIFT = 2.0
MIN_INTERNAL_EDGE_RATIO = 0.5
RISKY_COMMUNITY_LIMIT = 5

print("Project root resolved")
print(f"igraph {ig.__version__} | NumPy {np.__version__} | Python {platform.python_version()}")

Project root resolved
igraph 1.0.0 | NumPy 2.5.1 | Python 3.12.1


## 1. Artifact contract

Schema xác định các output chính của Community Detection, fraud EDA và model ablation.


In [2]:
RESULT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Sprint 4 community detection results",
    "type": "object",
    "required": [
        "schema_version", "data_contract", "community_config", "full_community",
        "fraud_eda", "model_ablation", "provenance",
    ],
    "properties": {
        "schema_version": {"type": "integer", "const": 1},
        "data_contract": {
            "type": "object",
            "required": [
                "data_sha256", "node_count", "directed_edge_count",
                "undirected_pair_count", "structural_arc_count",
            ],
        },
        "community_config": {
            "type": "object",
            "required": [
                "algorithm", "backend", "objective", "resolution", "seed",
                "iterations", "graph_policy",
            ],
        },
        "full_community": {"type": "object"},
        "fraud_eda": {"type": "object"},
        "model_ablation": {"type": "object"},
        "provenance": {"type": "object"},
    },
}

ASSIGNMENT_ARTIFACT_SPEC = {
    "path": "artifacts/metrics/sprint4_community_assignments.npz",
    "format": "npz",
    "arrays": {
        "node_id": {"dtype": "int64", "shape": ["node_count"]},
        "community_id": {"dtype": "int64", "shape": ["node_count"]},
    },
    "required_metadata": [
        "data_sha256", "graph_policy", "algorithm", "backend", "resolution",
        "seed", "iterations", "node_count", "undirected_pair_count",
    ],
}

display(pd.DataFrame([
    {"artifact": RESULT_PATH.name, "purpose": "metric/config theo từng giai đoạn"},
    {"artifact": SCHEMA_PATH.name, "purpose": "schema kiểm tra file kết quả"},
    {"artifact": Path(ASSIGNMENT_ARTIFACT_SPEC["path"]).name, "purpose": "node_id → community_id"},
]))

,artifact,purpose
0,sprint4_community_results.json,metric/config theo từng giai đoạn
1,sprint4_community_results.schema.json,schema kiểm tra file kết quả
2,sprint4_community_assignments.npz,node_id → community_id


## 2. Utilities


In [3]:
def file_sha256(path: str | Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_load(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def json_dump(path: str | Path, payload: dict) -> None:
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )


class PeakRSSMonitor(AbstractContextManager):
    def __init__(self, interval_seconds: float = 0.05):
        self.interval_seconds = interval_seconds
        self.process = psutil.Process()
        self.start_rss = 0
        self.peak_rss = 0
        self._stop = threading.Event()
        self._thread = None

    def _sample(self) -> None:
        while not self._stop.wait(self.interval_seconds):
            self.peak_rss = max(self.peak_rss, self.process.memory_info().rss)

    def __enter__(self):
        self.start_rss = self.process.memory_info().rss
        self.peak_rss = self.start_rss
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.peak_rss = max(self.peak_rss, self.process.memory_info().rss)
        self._stop.set()
        self._thread.join()
        return False

    @property
    def start_mib(self) -> float:
        return self.start_rss / 1024**2

    @property
    def peak_mib(self) -> float:
        return self.peak_rss / 1024**2


def build_undirected_pair_keys(edge_index: np.ndarray, num_nodes: int) -> np.ndarray:
    if edge_index.ndim != 2 or edge_index.shape[1] != 2:
        raise ValueError(f"Expected edge_index [E, 2], got {edge_index.shape}")
    source, target = edge_index[:, 0], edge_index[:, 1]
    if np.any(source == target):
        raise ValueError("Sprint 1 contract expected zero self-loop")
    low = np.minimum(source, target)
    high = np.maximum(source, target)
    return np.unique(low * np.int64(num_nodes) + high)


def summarize_sizes(membership: np.ndarray) -> dict[str, float | int]:
    sizes = np.bincount(membership)
    return {
        "community_count": int(sizes.size),
        "min": int(sizes.min()),
        "median": float(np.median(sizes)),
        "p90": float(np.quantile(sizes, 0.90)),
        "p99": float(np.quantile(sizes, 0.99)),
        "max": int(sizes.max()),
        "singleton_count": int(np.count_nonzero(sizes == 1)),
    }

## 3. Kiểm tra input và dựng unique undirected pairs

Đây là precondition tối thiểu trước full Leiden; EDA graph cơ bản đã được thực hiện ở Sprint 1.


In [4]:
contract_started = time.perf_counter()
data_sha256 = file_sha256(DATA_PATH)
if data_sha256 != EXPECTED_DATA_SHA256:
    raise AssertionError(f"Dataset SHA-256 mismatch: {data_sha256}")

with np.load(DATA_PATH, allow_pickle=False) as archive:
    edge_index = np.asarray(archive["edge_index"], dtype=np.int64)
    node_count = int(archive["y"].shape[0])

if node_count != EXPECTED_NODE_COUNT:
    raise AssertionError(f"Node count mismatch: {node_count:,}")
if edge_index.shape[0] != EXPECTED_DIRECTED_EDGE_COUNT:
    raise AssertionError(f"Directed edge count mismatch: {edge_index.shape[0]:,}")

pair_keys = build_undirected_pair_keys(edge_index, node_count)
undirected_pair_count = int(pair_keys.size)
structural_arc_count = undirected_pair_count * 2
if structural_arc_count != EXPECTED_STRUCTURAL_ARC_COUNT:
    raise AssertionError(
        f"structural_coalesced mismatch: {structural_arc_count:,} != "
        f"{EXPECTED_STRUCTURAL_ARC_COUNT:,}"
    )

data_contract = {
    "status": "passed",
    "data_sha256": data_sha256,
    "node_count": node_count,
    "directed_edge_count": int(edge_index.shape[0]),
    "undirected_pair_count": undirected_pair_count,
    "structural_arc_count": structural_arc_count,
    "elapsed_seconds": time.perf_counter() - contract_started,
}
del edge_index
display(pd.DataFrame([data_contract]))

community_config = {
    "algorithm": "leiden", "backend": "igraph", "backend_version": ig.__version__,
    "objective": "modularity", "resolution": LEIDEN_RESOLUTION,
    "seed": COMMUNITY_SEED, "iterations": LEIDEN_ITERATIONS,
    "graph_policy": "structural_coalesced_as_unique_undirected_pairs",
    "edge_weighting": "unweighted",
}
results = json_load(RESULT_PATH) if RESULT_PATH.exists() else {}
results.update({
    "schema_version": 1,
    "data_contract": data_contract,
    "community_config": community_config,
})
results.setdefault("full_community", {"status": "pending"})
results.setdefault("fraud_eda", {"status": "pending"})
results.setdefault("model_ablation", {"status": "pending"})
results.setdefault("provenance", {})
results["provenance"].update({
    "python": platform.python_version(), "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__, "psutil": psutil.__version__,
    "igraph": ig.__version__, "updated_at": datetime.now().astimezone().isoformat(),
})


,status,data_sha256,node_count,directed_edge_count,undirected_pair_count,structural_arc_count,elapsed_seconds
0,passed,95470dab2c48523f7118a92204c090de37a957bb053bd5...,3700550,4300999,3997260,7994520,3.063228


## 4. Leiden toàn graph và structural EDA

Assignment chỉ dùng graph structure và được khóa trước khi đọc nhãn. `coverage` là tỷ lệ cạnh nội bộ; `internal_edge_ratio = 2*internal_edges/volume`; `conductance` dùng cut chia cho volume nhỏ hơn của hai phía.


In [5]:
full_source = (pair_keys // np.int64(node_count)).astype(np.int64, copy=False)
full_target = (pair_keys % np.int64(node_count)).astype(np.int64, copy=False)
assignment_reused = False
graph_build_seconds = full_leiden_seconds = 0.0
can_reuse = REUSE_LOCKED_ASSIGNMENT and ASSIGNMENT_PATH.exists() and ASSIGNMENT_MANIFEST_PATH.exists()
if can_reuse:
    assignment_manifest = json_load(ASSIGNMENT_MANIFEST_PATH)
    manifest_matches = (
        assignment_manifest.get("data_sha256") == data_sha256
        and assignment_manifest.get("graph_policy") == community_config["graph_policy"]
        and assignment_manifest.get("algorithm") == "leiden"
        and assignment_manifest.get("backend") == "igraph"
        and assignment_manifest.get("resolution") == LEIDEN_RESOLUTION
        and assignment_manifest.get("seed") == COMMUNITY_SEED
        and assignment_manifest.get("iterations") == LEIDEN_ITERATIONS
        and assignment_manifest.get("node_count") == node_count
        and assignment_manifest.get("undirected_pair_count") == undirected_pair_count
    )
    if not manifest_matches:
        raise AssertionError("Assignment manifest không khớp dữ liệu/cấu hình")
    with np.load(ASSIGNMENT_PATH, allow_pickle=False) as saved:
        saved_node_ids = np.asarray(saved["node_id"], dtype=np.int64)
        full_membership = np.asarray(saved["community_id"], dtype=np.int64)
    if not np.array_equal(saved_node_ids, np.arange(node_count, dtype=np.int64)):
        raise AssertionError("node_id trong assignment không đúng thứ tự")
    if file_sha256(ASSIGNMENT_PATH) != assignment_manifest["assignment_sha256"]:
        raise AssertionError("Assignment SHA-256 không khớp manifest")
    del saved_node_ids
    assignment_reused = True
else:
    ig.set_random_number_generator(random.Random(COMMUNITY_SEED))
    with PeakRSSMonitor(interval_seconds=0.10) as monitor:
        started = time.perf_counter()
        full_edges = np.column_stack((full_source, full_target))
        full_graph = ig.Graph(n=node_count, edges=full_edges, directed=False)
        graph_build_seconds = time.perf_counter() - started
        started = time.perf_counter()
        partition = full_graph.community_leiden(
            objective_function="modularity", resolution=LEIDEN_RESOLUTION,
            n_iterations=LEIDEN_ITERATIONS,
        )
        full_leiden_seconds = time.perf_counter() - started
        full_membership = np.asarray(partition.membership, dtype=np.int64)
    full_runtime = {
        "graph_build_seconds": graph_build_seconds, "leiden_seconds": full_leiden_seconds,
        "rss_start_mib": monitor.start_mib, "rss_peak_mib": monitor.peak_mib,
        "rss_delta_mib": monitor.peak_mib - monitor.start_mib,
    }
    del full_edges, full_graph, partition
    gc.collect()
if full_membership.shape != (node_count,) or full_membership.min() != 0:
    raise AssertionError("Leiden membership không hợp lệ")
community_count = int(full_membership.max()) + 1
if np.unique(full_membership).size != community_count:
    raise AssertionError("Community ID không liên tục")
print(f"Assignment reused: {assignment_reused} | communities: {community_count:,}")
print(f"Graph build: {graph_build_seconds:.2f}s | Leiden: {full_leiden_seconds:.2f}s")

Assignment reused: True | communities: 1,739
Graph build: 0.00s | Leiden: 0.00s


In [6]:
community_sizes = np.bincount(full_membership, minlength=community_count).astype(np.int64)
source_community = full_membership[full_source]
target_community = full_membership[full_target]
internal_edge_mask = source_community == target_community
internal_edge_count = np.bincount(source_community[internal_edge_mask], minlength=community_count)
crossing = ~internal_edge_mask
boundary_edge_count = (
    np.bincount(source_community[crossing], minlength=community_count)
    + np.bincount(target_community[crossing], minlength=community_count)
)
volume = 2 * internal_edge_count + boundary_edge_count
total_volume = np.int64(2 * undirected_pair_count)
denominator = np.minimum(volume, total_volume - volume)
conductance = np.divide(
    boundary_edge_count, denominator,
    out=np.zeros(community_count, dtype=np.float64), where=denominator > 0,
)
internal_edge_ratio = np.divide(
    2 * internal_edge_count, volume,
    out=np.zeros(community_count, dtype=np.float64), where=volume > 0,
)
coverage = float(internal_edge_count.sum() / undirected_pair_count)
modularity = float(np.sum(
    internal_edge_count / undirected_pair_count
    - LEIDEN_RESOLUTION * np.square(volume / total_volume)
))
degree = np.bincount(full_source, minlength=node_count) + np.bincount(full_target, minlength=node_count)
internal_degree = np.zeros(node_count, dtype=np.int64)
np.add.at(internal_degree, full_source[internal_edge_mask], 1)
np.add.at(internal_degree, full_target[internal_edge_mask], 1)
community_table = pd.DataFrame({
    "community_id": np.arange(community_count), "size": community_sizes,
    "internal_edge_count": internal_edge_count, "boundary_edge_count": boundary_edge_count,
    "volume": volume, "conductance": conductance,
    "internal_edge_ratio": internal_edge_ratio,
})
size_summary = summarize_sizes(full_membership)
size_summary.update({
    "singleton_rate": float(size_summary["singleton_count"] / community_count),
    "largest_community_fraction": float(community_sizes.max() / node_count),
    "communities_size_le_5": int(np.count_nonzero(community_sizes <= 5)),
    "communities_size_le_5_rate": float(np.mean(community_sizes <= 5)),
})
non_isolated = volume > 0
structural_summary = {
    "modularity": modularity, "coverage": coverage,
    "internal_edge_ratio_global": coverage,
    "conductance_median_non_isolated": float(np.median(conductance[non_isolated])),
    "conductance_p90_non_isolated": float(np.quantile(conductance[non_isolated], 0.90)),
    "community_internal_edge_ratio_median_non_isolated": float(np.median(internal_edge_ratio[non_isolated])),
    "isolated_community_count": int(np.count_nonzero(~non_isolated)),
}

In [7]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
figure_01_path = FIGURE_DIR / "01_community_size_distribution.png"
positive_sizes = community_sizes[community_sizes > 0]
max_size = int(positive_sizes.max())
bins = np.unique(np.append(
    np.logspace(0, np.log10(max_size + 1), 55).astype(np.int64), max_size + 1
))
if bins.size < 2:
    bins = np.array([1, 2])
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
axes[0].hist(positive_sizes, bins=bins, color="#4C78A8", edgecolor="white")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Kích thước community (log)"); axes[0].set_ylabel("Số community (log)")
axes[0].set_title("Histogram")
ordered = np.sort(positive_sizes)
ccdf = (ordered.size - np.arange(ordered.size)) / ordered.size
axes[1].step(ordered, ccdf, where="post", color="#F58518", linewidth=2)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("Kích thước community ≥ x (log)"); axes[1].set_ylabel("P(Size ≥ x) (log)")
axes[1].set_title("CCDF")
fig.suptitle("Leiden toàn graph — phân bố kích thước community", fontsize=14)
fig.tight_layout(); fig.savefig(figure_01_path, dpi=200, bbox_inches="tight"); plt.close(fig)

if WRITE_ARTIFACTS and not assignment_reused:
    np.savez_compressed(
        ASSIGNMENT_PATH, node_id=np.arange(node_count, dtype=np.int64),
        community_id=full_membership,
    )
    assignment_manifest = {
        "schema_version": 1, "data_sha256": data_sha256,
        "graph_policy": community_config["graph_policy"], "algorithm": "leiden",
        "backend": "igraph", "backend_version": ig.__version__, "objective": "modularity",
        "resolution": LEIDEN_RESOLUTION, "seed": COMMUNITY_SEED,
        "iterations": LEIDEN_ITERATIONS, "node_count": node_count,
        "undirected_pair_count": undirected_pair_count, "community_count": community_count,
        "runtime": full_runtime, "created_at": datetime.now().astimezone().isoformat(),
    }
    assignment_manifest["assignment_sha256"] = file_sha256(ASSIGNMENT_PATH)
    json_dump(ASSIGNMENT_MANIFEST_PATH, assignment_manifest)
if WRITE_ARTIFACTS:
    community_table.to_csv(COMMUNITY_TABLE_PATH, index=False, compression="gzip")
results["full_community"] = {
    "status": "complete", "assignment_reused": assignment_reused,
    "assignment_path": ASSIGNMENT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "assignment_manifest_path": ASSIGNMENT_MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "assignment_sha256": assignment_manifest["assignment_sha256"],
    "community_table_path": COMMUNITY_TABLE_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "figure_01_path": figure_01_path.relative_to(PROJECT_ROOT).as_posix(),
    "runtime": assignment_manifest["runtime"], "community_sizes": size_summary,
    "structural_metrics": structural_summary,
    "metric_definitions": {
        "coverage": "internal_edges / all_undirected_edges",
        "internal_edge_ratio": "2 * internal_edges / community_volume",
        "conductance": "boundary / min(volume, total_volume-volume); 0 if volume=0",
    },
}
results["stage2_status"] = "complete"
results["provenance"]["updated_at"] = datetime.now().astimezone().isoformat()
if WRITE_ARTIFACTS:
    json_dump(RESULT_PATH, results)
display(pd.json_normalize({
    "sizes": size_summary, "metrics": structural_summary,
    "runtime": assignment_manifest["runtime"],
}).T.rename(columns={0: "value"}))
print(f"Assignment locked before label EDA: {ASSIGNMENT_PATH.relative_to(PROJECT_ROOT)}")

,value
sizes.community_count,1739.000000
sizes.min,99.000000
sizes.median,1964.000000
sizes.p90,2705.200000
sizes.p99,6853.960000
sizes.max,16609.000000
sizes.singleton_count,0.000000
sizes.singleton_rate,0.000000
sizes.largest_community_fraction,0.004488
sizes.communities_size_le_5,0.000000


Assignment locked before label EDA: artifacts\metrics\sprint4_community_assignments.npz


![Community-size distribution](../artifacts/figures/sprint4/01_community_size_distribution.png)


## 5. Fraud concentration và risky communities

Rule được khóa bằng train label trước validation: size ≥ 20, train labeled ≥ 20, train fraud ≥ 2, train fraud lift ≥ 2 và internal-edge ratio ≥ 0,5. Test label không được đọc.


In [8]:
with np.load(DATA_PATH, allow_pickle=False) as label_archive:
    labels = np.asarray(label_archive["y"], dtype=np.int64)
    train_indices = np.asarray(label_archive["train_mask"], dtype=np.int64)
    valid_indices = np.asarray(label_archive["valid_mask"], dtype=np.int64)
if not np.isin(labels[train_indices], [0, 1]).all():
    raise AssertionError("Train split chứa nhãn ngoài 0/1")
if not np.isin(labels[valid_indices], [0, 1]).all():
    raise AssertionError("Validation split chứa nhãn ngoài 0/1")

def labeled_fraud_counts(indices):
    communities = full_membership[indices]
    split_labels = labels[indices]
    labeled = np.bincount(communities, minlength=community_count).astype(np.int64)
    fraud = np.bincount(communities[split_labels == 1], minlength=community_count).astype(np.int64)
    rate = np.divide(fraud, labeled, out=np.full(community_count, np.nan), where=labeled > 0)
    return labeled, fraud, rate, float(np.mean(split_labels == 1))

train_labeled_count, train_fraud_count, train_fraud_rate, train_global_fraud_rate = labeled_fraud_counts(train_indices)
train_fraud_lift = train_fraud_rate / train_global_fraud_rate
community_table["train_labeled_count"] = train_labeled_count
community_table["train_fraud_count"] = train_fraud_count
community_table["train_fraud_rate"] = train_fraud_rate
community_table["train_fraud_lift"] = train_fraud_lift
risky_mask = (
    (community_sizes >= MIN_COMMUNITY_SIZE)
    & (train_labeled_count >= MIN_LABELED_COMMUNITY)
    & (train_fraud_count >= MIN_TRAIN_FRAUD_COUNT)
    & (train_fraud_lift >= MIN_TRAIN_FRAUD_LIFT)
    & (internal_edge_ratio >= MIN_INTERNAL_EDGE_RATIO)
)
ranked = community_table.loc[risky_mask].sort_values(
    ["train_fraud_lift", "train_fraud_count", "internal_edge_ratio", "community_id"],
    ascending=[False, False, False, True], kind="mergesort",
)
selected_train_only = ranked.head(RISKY_COMMUNITY_LIMIT).copy()
selected_community_ids = selected_train_only["community_id"].to_numpy(dtype=np.int64)
risk_rule = {
    "selection_labels": "train_only", "minimum_community_size": MIN_COMMUNITY_SIZE,
    "minimum_train_labeled_count": MIN_LABELED_COMMUNITY,
    "minimum_train_fraud_count": MIN_TRAIN_FRAUD_COUNT,
    "minimum_train_fraud_lift": MIN_TRAIN_FRAUD_LIFT,
    "minimum_internal_edge_ratio": MIN_INTERNAL_EDGE_RATIO,
    "ranking": ["train_fraud_lift desc", "train_fraud_count desc", "internal_edge_ratio desc", "community_id asc"],
    "maximum_selected": RISKY_COMMUNITY_LIMIT,
}
selection_payload = {
    "schema_version": 1, "data_sha256": data_sha256,
    "assignment_sha256": assignment_manifest["assignment_sha256"], "rule": risk_rule,
    "eligible_count": int(risky_mask.sum()),
    "selected_community_ids": selected_community_ids.tolist(),
    "selected_train_metrics": selected_train_only.replace({np.nan: None}).to_dict(orient="records"),
    "validation_labels_used_for_selection": False, "test_labels_used": False,
    "created_at": datetime.now().astimezone().isoformat(),
}
if WRITE_ARTIFACTS:
    json_dump(RISK_SELECTION_PATH, selection_payload)
print(f"Eligible train-only: {int(risky_mask.sum()):,}")
print(f"Locked IDs before validation: {selected_community_ids.tolist()}")

Eligible train-only: 51
Locked IDs before validation: [54, 286, 110, 296, 1544]


In [9]:
valid_labeled_count, valid_fraud_count, valid_fraud_rate, valid_global_fraud_rate = labeled_fraud_counts(valid_indices)
valid_fraud_lift = valid_fraud_rate / valid_global_fraud_rate
community_table["valid_labeled_count"] = valid_labeled_count
community_table["valid_fraud_count"] = valid_fraud_count
community_table["valid_fraud_rate"] = valid_fraud_rate
community_table["valid_fraud_lift"] = valid_fraud_lift
community_table["is_risky_train_rule"] = risky_mask
community_table["is_selected_risky"] = np.isin(community_table["community_id"], selected_community_ids)
columns = [
    "community_id", "size", "internal_edge_count", "boundary_edge_count",
    "conductance", "internal_edge_ratio", "train_labeled_count", "train_fraud_count",
    "train_fraud_rate", "train_fraud_lift", "valid_labeled_count", "valid_fraud_count",
    "valid_fraud_rate", "valid_fraud_lift",
]
risky_communities = community_table.loc[community_table["is_selected_risky"], columns].sort_values(
    "train_fraud_lift", ascending=False, kind="mergesort"
).copy()
top_node_ids = []
for community_id in risky_communities["community_id"].to_numpy(dtype=np.int64):
    nodes = np.flatnonzero(full_membership == community_id)
    order = np.lexsort((nodes, -internal_degree[nodes]))
    top_node_ids.append(";".join(map(str, nodes[order[:10]].tolist())))
risky_communities["top_node_ids_by_internal_degree"] = top_node_ids
risky_communities["model_probability_status"] = "pending_stage4"

selected_valid_labeled = int(valid_labeled_count[selected_community_ids].sum())
selected_valid_fraud = int(valid_fraud_count[selected_community_ids].sum())
selected_valid_rate = selected_valid_fraud / selected_valid_labeled if selected_valid_labeled else None
selected_valid_lift = selected_valid_rate / valid_global_fraud_rate if selected_valid_rate is not None else None
has_validation = valid_labeled_count[selected_community_ids] > 0
individual_lifts = valid_fraud_lift[selected_community_ids][has_validation]
above_one = float(np.mean(individual_lifts > 1)) if individual_lifts.size else None
train_fraud_captured = int(train_fraud_count[selected_community_ids].sum())
stable_communities = community_table[
    (community_table["train_labeled_count"] >= MIN_LABELED_COMMUNITY)
    & (community_table["valid_labeled_count"] >= MIN_LABELED_COMMUNITY)
].copy()
train_rate_rank = stable_communities["train_fraud_rate"].rank(method="average").to_numpy()
valid_rate_rank = stable_communities["valid_fraud_rate"].rank(method="average").to_numpy()
rate_spearman = float(np.corrcoef(train_rate_rank, valid_rate_rank)[0, 1])
fraud_rate_stability = {
    "minimum_labels_per_split": MIN_LABELED_COMMUNITY,
    "community_count": int(len(stable_communities)),
    "spearman_train_validation": rate_spearman,
}
validation_check = {
    "global_fraud_rate": valid_global_fraud_rate,
    "selected_labeled_count": selected_valid_labeled,
    "selected_fraud_count": selected_valid_fraud,
    "selected_fraud_rate": selected_valid_rate,
    "selected_fraud_lift": selected_valid_lift,
    "selected_with_at_least_one_validation_label": int(has_validation.sum()),
    "fraction_selected_with_validation_lift_above_1": above_one,
}
concentration_summary = {
    "train_global_fraud_rate": train_global_fraud_rate,
    "train_total_fraud_count": int(train_fraud_count.sum()),
    "train_fraud_in_selected": train_fraud_captured,
    "train_fraud_capture_rate_selected": float(train_fraud_captured / train_fraud_count.sum()),
    "validation": validation_check,
    "train_validation_stability": fraud_rate_stability,
}
if WRITE_ARTIFACTS:
    community_table.to_csv(COMMUNITY_TABLE_PATH, index=False, compression="gzip")
    risky_communities.to_csv(RISKY_TABLE_PATH, index=False)
display(risky_communities.round(4))
display(pd.json_normalize(concentration_summary).T.rename(columns={0: "value"}))

,community_id,size,internal_edge_count,boundary_edge_count,conductance,internal_edge_ratio,train_labeled_count,train_fraud_count,train_fraud_rate,train_fraud_lift,valid_labeled_count,valid_fraud_count,valid_fraud_rate,valid_fraud_lift,top_node_ids_by_internal_degree,model_probability_status
54,54,4424,4871,93,0.0095,0.9905,731,45,0.0616,4.8643,169,8,0.0473,3.7418,1490390;1942260;2194380;1277295;1869349;342044...,pending_stage4
286,286,2059,2262,13,0.0029,0.9971,383,16,0.0418,3.3010,72,6,0.0833,6.5872,1614611;2792461;2659840;407054;614407;1097450;...,pending_stage4
110,110,787,852,1,0.0006,0.9994,174,7,0.0402,3.1789,31,0,0.0000,0.0000,2809087;1542027;2619958;86599;545971;1343854;2...,pending_stage4
296,296,1598,1713,22,0.0064,0.9936,401,16,0.0399,3.1528,85,4,0.0471,3.7198,149487;1908192;1742591;25571;3258261;1827102;3...,pending_stage4
1544,1544,338,352,1,0.0014,0.9986,82,3,0.0366,2.8909,11,0,0.0000,0.0000,2492742;1460791;3163686;919064;1393522;2066218...,pending_stage4


,value
train_global_fraud_rate,0.012655
train_total_fraud_count,10857.000000
train_fraud_in_selected,87.000000
train_fraud_capture_rate_selected,0.008013
validation.global_fraud_rate,0.012651
validation.selected_labeled_count,368.000000
validation.selected_fraud_count,18.000000
validation.selected_fraud_rate,0.048913
validation.selected_fraud_lift,3.866402
validation.selected_with_at_least_one_validation_label,5.000000


In [10]:
figure_02_path = FIGURE_DIR / "02_fraud_concentration_by_community.png"
stable_train = community_table[community_table["train_labeled_count"] >= MIN_LABELED_COMMUNITY]
top_fraud = stable_train.nlargest(15, ["train_fraud_count", "train_fraud_lift"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].bar(top_fraud["community_id"].astype(str), top_fraud["train_fraud_count"], color="#E45756")
axes[0].tick_params(axis="x", rotation=70)
axes[0].set(
    xlabel="Community ID", ylabel="Train fraud count",
    title="15 community có nhiều train fraud nhất",
)

scatter_frame = stable_train
if len(scatter_frame) > 50_000:
    scatter_frame = scatter_frame.sample(50_000, random_state=COMMUNITY_SEED)
points = axes[1].scatter(
    scatter_frame["train_labeled_count"], scatter_frame["train_fraud_rate"],
    c=np.clip(scatter_frame["train_fraud_lift"], 0, 10),
    s=np.clip(np.sqrt(scatter_frame["size"]) * 2, 5, 80),
    cmap="viridis", alpha=0.45, linewidths=0,
)
axes[1].set_xscale("log")
axes[1].axhline(
    train_global_fraud_rate, color="black", linestyle="--", linewidth=1,
    label="Train global rate",
)
axes[1].set(
    xlabel="Train labeled count — trục log", ylabel="Train fraud rate",
    title="Fraud rate, lift và community size",
)
axes[1].legend()
fig.colorbar(points, ax=axes[1], label="Train fraud lift (clip 10)")

fig.suptitle("Fraud count, rate và lift theo community (không dùng test)", fontsize=14)
fig.tight_layout()
fig.savefig(figure_02_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {figure_02_path.relative_to(PROJECT_ROOT)}")


Saved: artifacts\figures\sprint4\02_fraud_concentration_by_community.png


![Fraud concentration by community](../artifacts/figures/sprint4/02_fraud_concentration_by_community.png)


In [11]:
fraud_eda_result = {
    "status": "complete",
    "label_policy": {
        "selection": "train_only", "validation": "evaluation_after_selection_lock",
        "test_labels_used": False,
    },
    "global_rates": {
        "train_fraud_rate": train_global_fraud_rate,
        "validation_fraud_rate": valid_global_fraud_rate,
    },
    "risk_rule": risk_rule, "eligible_community_count": int(risky_mask.sum()),
    "selected_community_ids": selected_community_ids.tolist(),
    "concentration": concentration_summary,
    "artifacts": {
        "community_table": COMMUNITY_TABLE_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "selection_lock": RISK_SELECTION_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "risky_community_table": RISKY_TABLE_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "figure_02": figure_02_path.relative_to(PROJECT_ROOT).as_posix(),
    },
}
results["fraud_eda"] = fraud_eda_result
results["stage3_status"] = "complete"
results["provenance"]["updated_at"] = datetime.now().astimezone().isoformat()
if WRITE_ARTIFACTS:
    json_dump(RESULT_PATH, results)
    print(f"Saved: {RESULT_PATH.relative_to(PROJECT_ROOT)}")
stage_summary = pd.DataFrame([
    {"stage": "Leiden + structural EDA", "status": results["stage2_status"]},
    {"stage": "Fraud concentration", "status": results["stage3_status"]},
    {"stage": "Community ablation", "status": results["model_ablation"]["status"]},
])
display(stage_summary)


Saved: artifacts\metrics\sprint4_community_results.json


,stage,status
0,Leiden + structural EDA,complete
1,Fraud concentration,complete
2,Community ablation,complete_final_test


**Độ ổn định train–validation.** Trên 1.704 community có ít nhất 20 label ở cả hai split, Spearman correlation giữa train và validation fraud rate là `0,089`. Tín hiệu fraud ở cấp community nhìn chung còn khá nhiễu, dù một số community top đầu vẫn có validation lift cao.


## 6. Tạo community features

Structural features gồm `log1p(size)`, `log1p(internal edges)`, internal density, node internal-degree ratio và conductance. Community risk chỉ dùng train label; train node dùng leave-one-out. Mean/std được fit trên train.


In [12]:
structural_feature_names = [
    "log_community_size", "log_internal_edge_count", "internal_density",
    "node_internal_degree_ratio", "community_conductance",
]
node_community = full_membership
node_size = community_sizes[node_community].astype(np.float64)
node_internal_edges = internal_edge_count[node_community].astype(np.float64)
density_denominator = node_size * np.maximum(node_size - 1, 1)
community_density = np.divide(
    2 * internal_edge_count, community_sizes * np.maximum(community_sizes - 1, 1),
    out=np.zeros(community_count, dtype=np.float64), where=community_sizes > 1,
)
node_internal_ratio = np.divide(
    internal_degree, degree, out=np.zeros(node_count, dtype=np.float64), where=degree > 0,
)
structural_raw = np.column_stack((
    np.log1p(node_size), np.log1p(node_internal_edges),
    community_density[node_community], node_internal_ratio,
    conductance[node_community],
)).astype(np.float32)

community_train_rate = np.divide(
    train_fraud_count, train_labeled_count,
    out=np.full(community_count, train_global_fraud_rate, dtype=np.float64),
    where=train_labeled_count >= MIN_LABELED_COMMUNITY,
)
risk_raw = community_train_rate[node_community].astype(np.float64)
train_communities = node_community[train_indices]
train_targets = labels[train_indices].astype(np.float64)
loo_count = train_labeled_count[train_communities] - 1
loo_fraud = train_fraud_count[train_communities] - train_targets
community_loo = np.divide(
    loo_fraud, loo_count, out=np.zeros_like(loo_fraud, dtype=np.float64), where=loo_count > 0,
)
global_loo = (train_fraud_count.sum() - train_targets) / (len(train_indices) - 1)
risk_raw[train_indices] = np.where(
    loo_count >= MIN_LABELED_COMMUNITY, community_loo, global_loo,
)

def train_only_standardize(values, train_nodes):
    mean = values[train_nodes].mean(axis=0, dtype=np.float64)
    std = values[train_nodes].std(axis=0, dtype=np.float64)
    safe_std = np.where(std > 0, std, 1.0)
    return ((values - mean) / safe_std).astype(np.float32), mean, safe_std

structural_features, structural_mean, structural_std = train_only_standardize(structural_raw, train_indices)
risk_feature_2d, risk_mean, risk_std = train_only_standardize(risk_raw[:, None], train_indices)
risk_feature = risk_feature_2d[:, 0]
if not np.isfinite(structural_features).all() or not np.isfinite(risk_feature).all():
    raise AssertionError("Community feature chứa NaN/Inf")

feature_contract = {
    "schema_version": 1, "data_sha256": data_sha256,
    "assignment_sha256": assignment_manifest["assignment_sha256"],
    "feature_path": COMMUNITY_FEATURE_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "structural_feature_names": structural_feature_names,
    "risk_feature_name": "train_only_community_fraud_rate_leave_one_out",
    "minimum_labeled_community": MIN_LABELED_COMMUNITY,
    "train_leave_one_out": True, "validation_test_use_train_labels_only": True,
    "standardization_fit": "train_nodes_only",
    "safe_for_pre_message_passing": False,
    "required_gnn_integration": "after_message_passing_seed_output_only",
    "leakage_note": (
        "Leave-one-out protects the seed feature only; pre-message-passing use lets "
        "the seed label return through same-community neighbor features."
    ),
    "structural_mean": structural_mean.tolist(), "structural_std": structural_std.tolist(),
    "risk_mean": float(risk_mean[0]), "risk_std": float(risk_std[0]),
    "shape": {"structural": list(structural_features.shape), "risk": list(risk_feature.shape)},
}
feature_artifact_reused = False
if COMMUNITY_FEATURE_PATH.exists() and COMMUNITY_FEATURE_MANIFEST_PATH.exists():
    existing_manifest = json_load(COMMUNITY_FEATURE_MANIFEST_PATH)
    contract_matches = all(
        existing_manifest.get(key) == value for key, value in feature_contract.items()
        if key not in {"structural_mean", "structural_std", "risk_mean", "risk_std"}
    )
    hash_matches = (
        existing_manifest.get("feature_sha256") == file_sha256(COMMUNITY_FEATURE_PATH)
    )
    if contract_matches and hash_matches:
        with np.load(COMMUNITY_FEATURE_PATH, allow_pickle=False) as saved:
            arrays_match = (
                np.array_equal(saved["node_id"], np.arange(node_count, dtype=np.int64))
                and np.array_equal(saved["structural_features"], structural_features)
                and np.array_equal(saved["community_risk_feature"], risk_feature)
            )
        if not arrays_match:
            raise AssertionError("Locked community feature differs from deterministic rebuild")
        feature_manifest = existing_manifest
        feature_artifact_reused = True

if not feature_artifact_reused:
    np.savez_compressed(
        COMMUNITY_FEATURE_PATH,
        node_id=np.arange(node_count, dtype=np.int64),
        structural_features=structural_features,
        community_risk_feature=risk_feature,
    )
    feature_manifest = {
        **feature_contract,
        "created_at": datetime.now().astimezone().isoformat(),
    }
    feature_manifest["feature_sha256"] = file_sha256(COMMUNITY_FEATURE_PATH)
    json_dump(COMMUNITY_FEATURE_MANIFEST_PATH, feature_manifest)

results["community_features"] = {
    "status": "complete", "artifact_reused": feature_artifact_reused, **feature_manifest
}
json_dump(RESULT_PATH, results)
del structural_raw, risk_feature_2d
print(
    f"{'Reused' if feature_artifact_reused else 'Saved'}: "
    f"{COMMUNITY_FEATURE_PATH.relative_to(PROJECT_ROOT)}"
)
display(pd.DataFrame({
    "feature": structural_feature_names + [feature_manifest["risk_feature_name"]],
    "train_mean_after_standardization": np.r_[structural_features[train_indices].mean(0), risk_feature[train_indices].mean()],
    "train_std_after_standardization": np.r_[structural_features[train_indices].std(0), risk_feature[train_indices].std()],
}))

Reused: artifacts\metrics\sprint4_community_features.npz


,feature,train_mean_after_standardization,train_std_after_standardization
0,log_community_size,-8.009223e-08,0.999681
1,log_internal_edge_count,1.755391e-07,0.999786
2,internal_density,4.722585e-08,0.999389
3,node_internal_degree_ratio,1.371527e-06,0.996414
4,community_conductance,-1.313652e-07,0.999586
5,train_only_community_fraud_rate_leave_one_out,-9.248839e-10,1.000000
